# VeriMem on Kaggle

Runs the whole pipeline inside one Kaggle session: the repo, the data, a local Qwen
teacher, and the baseline run. Nothing here talks to your Mac.

**Before running anything, set these in the right-hand sidebar:**

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Internet | **On** (needs phone verification once, under Settings) |
| Persistence | Files only |

**Attach the data**: Add Input -> Datasets -> search your uploaded `verimem-data`.
Without it, cell 4 rebuilds the evidence pool from scratch, which takes ~100 minutes.

Kaggle gives you 30 GPU hours a week and kills a session after 12 hours. Everything
written outside `/kaggle/working` disappears when the session ends.

## 1. What machine did we get?

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working | tail -1
!free -g | head -2
import torch

print(
    "torch",
    torch.__version__,
    "| cuda",
    torch.cuda.is_available(),
    "| bf16",
    torch.cuda.is_bf16_supported(),
)
# T4 has no bf16. core/device.py probes for this and falls back to fp16 on its own.

## 2. Get the code

In [ ]:
import os
import pathlib

REPO = pathlib.Path("/kaggle/working/VeriMem")
if not REPO.exists():
    !git clone -q https://github.com/TERABYTE05/VeriMem.git {REPO}
%cd {REPO}
!git pull -q
os.chdir(REPO)
import sys

sys.path.insert(0, str(REPO))
!git log --oneline -1

## 3. Install dependencies

Everything the baseline needs. Training deps are at the bottom of the
notebook, deliberately out of the way — they are slow and not needed yet.

In [ ]:
# Only what the baseline run needs. Seconds, not minutes.
!pip install -q openai pyyaml python-dotenv 2>&1 | tail -1
import openai

print("ready:", openai.__version__)

## 4. Data

Two things are needed: the AVeriTeC claims (12 MB, re-downloaded each session) and the
evidence pool (7.9 MB, from your Kaggle Dataset).

**The pool took 104 minutes to build.** If the dataset is not attached this cell
rebuilds it, so attach it rather than waiting.

In [ ]:
import os
import pathlib
import shutil
import subprocess
import zipfile

DATA = pathlib.Path("/kaggle/working/data/averitec")

DATA.mkdir(parents=True, exist_ok=True)

os.environ["VERIMEM_DATA_DIR"] = "/kaggle/working/data"

os.environ["VERIMEM_CACHE_DIR"] = "/kaggle/working/cache"

os.environ["VERIMEM_RESULTS_DIR"] = "/kaggle/working/results"



WANTED = ("dev.json", "train.json", "misinfo_list.txt", "pool_dev.jsonl")

# Restore cached model calls so work already done is never repeated (rule 1).
CACHE = pathlib.Path("/kaggle/working/cache")
CACHE.mkdir(parents=True, exist_ok=True)
_cached = pathlib.Path("/kaggle/input").rglob("cache/llm") if pathlib.Path("/kaggle/input").exists() else []
for src_cache in _cached:
    if src_cache.is_dir() and not (CACHE / "llm").exists():
        shutil.copytree(src_cache, CACHE / "llm")
        n = len(list((CACHE / "llm").rglob("*.json")))
        print(f"restored {n} cached model calls")
        break

INPUT = pathlib.Path("/kaggle/input")



# --- say what is actually attached, before trying anything -------------------------

print("=== /kaggle/input ===")

listing = sorted(INPUT.rglob("*"))[:25] if pathlib.Path("/kaggle/input").exists() else []

for p in listing:

    print(f"   {p}" + (f"  {p.stat().st_size / 1e6:.1f} MB" if p.is_file() else "/"))

if not listing:

    print("   (nothing attached — use Add Input -> Datasets in the right sidebar)")



# --- take the files wherever they are: loose, nested, or still zipped --------------

for name in WANTED:

    if (DATA / name).exists():

        continue

    for src in INPUT.rglob(name):

        shutil.copy(src, DATA / name)

        print(f"copied {name} ({src.stat().st_size / 1e6:.1f} MB)")

        break

for z in INPUT.rglob("*.zip") if pathlib.Path("/kaggle/input").exists() else []:

    with zipfile.ZipFile(z) as zf:

        for member in zf.namelist():

            name = pathlib.Path(member).name

            if name in WANTED and not (DATA / name).exists():

                (DATA / name).write_bytes(zf.read(member))

                print(f"unzipped {name}")



# --- only now fall back to the network, and never hang on it ----------------------

missing = [f for f in ("dev.json", "train.json", "misinfo_list.txt") if not (DATA / f).exists()]

if missing:

    online = subprocess.run(

        [

            "curl",

            "-sS",

            "--max-time",

            "10",

            "-o",

            "/dev/null",

            "-w",

            "%{http_code}",

            "https://raw.githubusercontent.com",

        ],

        capture_output=True,

        text=True,

    ).stdout.strip()

    if not online.startswith(("2", "3")):

        raise SystemExit(

            "No internet. Turn Internet ON in the right sidebar "

            "(Settings -> Phone Verification first if the switch is greyed out)."

        )

    BASE = "https://raw.githubusercontent.com/MichSchli/AVeriTeC/main/data"

    for f in missing:

        subprocess.run(

            ["curl", "-sSL", "--max-time", "180", "-o", str(DATA / f), f"{BASE}/{f}"], check=True

        )

        print("downloaded", f)



# --- report, do not silently proceed without the pool -----------------------------

print("\n=== data/averitec ===")

for f in sorted(DATA.iterdir()):

    print(f"   {f.name:<22} {f.stat().st_size / 1e6:7.1f} MB")



if (DATA / "pool_dev.jsonl").exists():

    from retrieval.store import PooledKnowledgeStore



    pool = PooledKnowledgeStore(DATA / "pool_dev.jsonl")

    print(f"\npool ready: {pool.n_documents} documents, {len(pool)} passages")

else:

    print("\n!! NO EVIDENCE POOL. Attach the verimem-data dataset and re-run this cell.")

    print("!! Rebuilding it here costs ~100 minutes:")

    print("!! !python -m retrieval.fetch_archive --data data/averitec/dev.json --workers 4")

## 5. The teacher model

Ollama runs as a local server inside this session and speaks the OpenAI API, so
`core/llm.py` talks to it with no code change. It is only reachable from this notebook.

First run downloads ~4.7 GB.

In [ ]:
import os
import shutil
import subprocess
import time
import urllib.request

# The official installer assumes systemd, which Kaggle containers do not have, and
# piping it to `tail` hides the failure. Try it, then fall back to the plain tarball:
# ollama is a single binary and needs no service manager to run in the foreground.
if shutil.which("ollama") is None:
    r = subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True, capture_output=True, text=True,
    )
    print((r.stdout or r.stderr)[-600:])

if shutil.which("ollama") is None:
    print("installer gave no binary; falling back to the tarball")
    r = subprocess.run(
        "curl -L# https://ollama.com/download/ollama-linux-amd64.tgz -o /tmp/o.tgz && "
        "tar -C /usr -xzf /tmp/o.tgz",
        shell=True, capture_output=True, text=True,
    )
    print(r.stderr[-400:] if r.returncode else "tarball extracted")

OLLAMA = shutil.which("ollama")
if OLLAMA is None:
    raise RuntimeError(
        "ollama could not be installed. Check Internet is ON in the right sidebar: "
        "with it off both downloads above fail silently."
    )
version = subprocess.run([OLLAMA, "--version"], capture_output=True, text=True).stdout
print("binary:", OLLAMA, "|", version.strip())

subprocess.Popen([OLLAMA, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(90):  # wait for readiness rather than guessing a sleep
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama serve did not come up within 90s")
print("server is up")

# ~4.7 GB on a first run. Later sessions can skip it by caching ~/.ollama into the dataset.
!ollama pull qwen2.5:7b

os.environ["VERIMEM_API_BASE_URL"] = "http://localhost:11434/v1"
os.environ["VERIMEM_API_KEY"] = "ollama"
os.environ["VERIMEM_API_MODEL"] = "qwen2.5:7b"

## 6. One real call, before anything expensive

In [ ]:
from core.llm import LLMClient

llm = LLMClient()
r = llm.complete("Reply with the single word: ok", max_tokens=8)
print(f"reply={r.text!r}  tokens={r.total_tokens}  cost=${r.cost_usd}  provider={llm.provider}")

## 7. The baseline run (P0.11)

20 claims, both baselines, against the real evidence pool. This is the G0 gate.
Writes `results/<date>_both-dev/` with config, metrics, notes and trajectories.

In [ ]:
# The whole dev split. Cached calls replay instantly; only new ones cost time.
# Start with --limit 20 if you want to check the wiring before committing to the full run.
!python -m eval.run_baselines \
    --data data/averitec/dev.json \
    --store data/averitec/pool_dev.jsonl \
    --system both --name dev-full --yes

## 8. Keep what matters

`/kaggle/working` is saved as notebook output; everything else vanishes. Download the
results, and re-upload the cache so the next session does not repeat these calls.

In [ ]:
!du -sh /kaggle/working/results /kaggle/working/cache 2>/dev/null
!ls -R /kaggle/working/results | head -20
# Then: Output tab (right) -> download. Re-upload cache/ into the verimem-data dataset
# so the next session reuses every model call instead of paying for it again.

### Training dependencies — skip this for now

`unsloth` pulls a specific torch/xformers combination and pip often spends **10+ minutes**
backtracking to resolve it. None of it is needed for the baseline run below.

Run this cell only when you get to training (P2.9 / P2.10), and expect it to be slow.

In [ ]:
!pip install -q peft trl datasets bitsandbytes 2>&1 | tail -2
!pip install -q unsloth unsloth_zoo 2>&1 | tail -2
import torch

print("bf16:", torch.cuda.is_bf16_supported(), "(T4 is False -> fp16, handled automatically)")

### Save the cache back

Zip it, download it from the Output panel, and replace `verimem-data` with the new
version. The next session then starts with every call already done.

In [ ]:
!cd /kaggle/working && zip -qr verimem-data.zip data/averitec cache/llm && ls -lh verimem-data.zip